# Baseline: test-set run + submission package (Colab)
Produces `matching_results.tsv` and `candidate_pairs.tsv` for **all 1,732,544 test S1 entities** using the saved baseline model
(`models/baseline_matcher.joblib`, threshold 0.65, validated at F0.5 = 0.8755 on the full pool), then validates them and builds the submission zip.

**Requirements**
- A **High-RAM** runtime (ideally 50 GB+ and 8+ CPUs). The test pool is ~10M records (India 4.7M, US 3.8M, France 1.4M); a standard 12 GB runtime is killed by the OS while indexing.
- In `MyDrive/MLChallenge/`: `colab_code.zip` (rebuilt, includes the model and validator) and the `dataset/` folder (needs `test/`).
- Expect **several hours**. Test mode is chunked (10k S1 per chunk) and **resumable**: chunk results are cached on Drive, so after a disconnect re-run the setup cells and the run cell and it skips finished chunks (the index build, a few minutes, is repeated).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
D=/content/drive/MyDrive/MLChallenge
rm -rf /content/work && mkdir -p /content/work/student_resource && cd /content/work
unzip -q $D/colab_code.zip
if [ -f $D/dataset.zip ]; then unzip -q $D/dataset.zip -d student_resource
elif [ -d $D/dataset ]; then cp -r $D/dataset student_resource/dataset; fi
if [ ! -d student_resource/dataset/test ] && [ -d student_resource/dataset/dataset/test ]; then
  mv student_resource/dataset/dataset/* student_resource/dataset/ && rmdir student_resource/dataset/dataset
fi
ls student_resource/dataset student_resource/dataset/test
pip -q install rapidfuzz xgboost pyyaml psutil
python -c "import sklearn; print('sklearn here:', sklearn.__version__, '(model was trained with 1.6.1)')"
nproc; free -g | head -2

In [ ]:
import os, re, subprocess, collections, shutil
os.chdir('/content/work')
mem_gb = int(open('/proc/meminfo').readline().split()[1]) / 1e6
NW = max(1, os.cpu_count() or 1)
print(f'RAM {mem_gb:.0f} GB, CPUs {NW}')
assert mem_gb >= 40, 'Need a High-RAM runtime (>=40 GB): the 10M-record test pool will be OOM-killed on a standard runtime.'
assert os.path.exists('student_resource/dataset/test/test_source1.tsv'), 'dataset path mismatch'

# keep the resumable chunk cache on Drive so a disconnect does not lose finished chunks
CACHE = '/content/drive/MyDrive/MLChallenge/test_cache'
os.makedirs(CACHE, exist_ok=True)
txt = open('configs/baseline.yaml').read()
txt, n = re.subn(r'cache_dir:\s*\S+', f'cache_dir: {CACHE}', txt, count=1)
assert n == 1
# smaller sparse-matmul batches lower each worker's peak memory (does not change the results)
txt, n2 = re.subn(r'(batch_size:\s*)\d+', r'\g<1>100', txt, count=1)
assert n2 == 1
open('configs/baseline.yaml', 'w').write(txt)
print(open('configs/baseline.yaml').read().split('candidates:')[1][:200])

## Test run
Uses `--threshold 0.65` explicitly (the dev metrics file is not present on Colab). Progress and ETA are printed per chunk.
Workers fork the fitted index, so a high worker count multiplies memory only for per-chunk data; lower `NW` if you see the memory guard trigger.

In [ ]:
# retry with fewer workers if memory runs out; finished chunks are cached on Drive, so a retry only redoes the missing ones
import time
def attempt(workers):
    cmd = ['python', '-u', '-m', 'src.pipeline', '--config', 'configs/baseline.yaml', '--mode', 'test',
           '--allow-full-test', '--resume', '--threshold', '0.65', '--workers', str(workers),
           '--chunk-size', '10000', '--no-log']
    print(' '.join(cmd), flush=True)
    os.makedirs('results', exist_ok=True)
    with open('results/test_run.log', 'a') as lf:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True); lf.write(line)
        return p.wait()

rc = None
for w in [min(NW, 6), 4, 3]:
    rc = attempt(w)
    print(f'\n=== attempt with {w} workers: returncode {rc} (0 = complete, 2 = incomplete coverage, negative/137 = killed) ===\n')
    if rc == 0:
        break
    time.sleep(5)
assert rc == 0, 'run is still incomplete: do NOT use these outputs. Re-run this cell (finished chunks are reused).'


## Validate with the competition's validator

In [ ]:
r = subprocess.run(['python3', 'utils/validate_submission.py',
                    '--matching', 'outputs/test/matching_results.tsv',
                    '--candidate', 'outputs/test/candidate_pairs.tsv',
                    '--test-dir', 'student_resource/dataset/test'], text=True, capture_output=True)
print(r.stdout[-4000:], r.stderr[-2000:])
assert r.returncode == 0, 'validator did not PASS: fix before submitting'

## Build the submission zip
Fill in `TEAM_NAME` and complete the TODOs in `Documentation_draft.md` **before** submitting (download it from `submission_extras/`, edit, re-upload,
or edit in the Colab file browser). The zip below copies the draft as `Documentation_template.md`; replace it once it is finished.

In [ ]:
TEAM_NAME = 'TEAM_NAME_HERE'   # <-- set this
import zipfile, pathlib
out = f'/content/drive/MyDrive/MLChallenge/{TEAM_NAME}_submission.zip'
root = 'code/business_entity_resolution'
def add_tree(z, src, dst):
    for p in pathlib.Path(src).rglob('*'):
        if p.is_file() and '__pycache__' not in p.parts and p.suffix != '.pyc':
            z.write(p, f'{dst}/{p.relative_to(src)}')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write('outputs/test/matching_results.tsv', 'output/matching_results.tsv')
    z.write('outputs/test/candidate_pairs.tsv', 'output/candidate_pairs.tsv')
    add_tree(z, 'src', f'{root}/src')
    add_tree(z, 'configs', f'{root}/configs')
    add_tree(z, 'models', f'{root}/models')
    z.write('submission_extras/README.md', f'{root}/README.md')
    z.write('requirements.txt', f'{root}/requirements.txt')
    z.write('submission_extras/Documentation_draft.md', 'Documentation_template.md')
print('wrote', out)
for f in ('matching_results.tsv', 'candidate_pairs.tsv'):   # also keep loose copies for the portal upload
    shutil.copy(f'outputs/test/{f}', f'/content/drive/MyDrive/MLChallenge/{f}')